In [12]:
!pip install --upgrade pip
!pip install numpy
!pip install pandas
!pip install ollama
!pip install tqdm
!pip install dotenv
!pip install torch
!pip install PIL 

ERROR: Could not find a version that satisfies the requirement PIL (from versions: none)
ERROR: No matching distribution found for PIL


In [17]:
import logging
import sys
import re
import json
from pathlib import Path

try:
    import numpy as np
except ModuleNotFoundError:
    import sys, subprocess, importlib
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy"])
    np = importlib.import_module("numpy")
import pandas as pd
import ollama
from tqdm.notebook import tqdm

sys.path.insert(0, str(Path.cwd().parent))

import config as cfg

from src.task2_embeddings_and_rag import (
    CLIPEncoder,
    ProductVectorStore,
    build_vector_store,
    evaluate_retrieval,
    retrieve,
)

from src.task3_llm_integration import (
    MultimodalChatbot,
    _format_product_context,
    few_shot_prompt,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger(__name__)

In [18]:
processed_path = cfg.PROCESSED_DIR / "products_processed.csv"

df = pd.read_csv(processed_path)

store, encoder = build_vector_store(df)

print(df.shape)
df.head()

INFO | Loading CLIP model 'ViT-B-32' (pretrained='openai') on cpu …
INFO | Parsing model identifier. Schema: None, Identifier: ViT-B-32
INFO | Loaded built-in ViT-B-32 model config.
INFO | HTTP Request: HEAD https://huggingface.co/timm/vit_base_patch32_clip_224.openai/resolve/main/open_clip_model.safetensors "HTTP/1.1 302 Found"
/Users/emandabisrat/Downloads/GRAD SCHOOL/GEN AI /Ecommerce-RAG/.venv/lib/python3.14/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
INFO | Instantiating model architecture: CLIP
INFO | Loading full pretrained weights from: /Users/emandabisrat/.cache/huggingface/hub/models--timm--vit_base_patch32_clip_224.openai/snapshots/a6f597a30f7b82c51704746581f9a4e41421e878/open_clip_model.safetensors
INFO | Final image preprocessing configuration set: {'size': (224, 224), 'mode': 'RGB', 'mean': (0.48145466, 0.4578275, 0.40821073), 'std': (0

(5000, 32)


,uniq_id,product_name,brand_name,asin,category,upc_ean_code,list_price,selling_price,quantity,model_number,...,color,ingredients,direction_to_use,is_amazon_seller,size_quantity_variant,product_description,price_usd,product_id,combined_description,local_image_path
0,4c69b61db1fc16e7013b43fc926e502d,"DB Longboards CoreFlex Crossbow 41"" Bamboo Fib...",NaN,NaN,Sports & Outdoors | Outdoor Recreation | Skate...,NaN,NaN,$237.68,NaN,NaN,...,NaN,NaN,NaN,Y,NaN,NaN,237.68,0,"Product: DB Longboards CoreFlex Crossbow 41"" B...",/Users/emandabisrat/Downloads/GRAD SCHOOL/GEN ...
1,66d49bbed043f5be260fa9f7fbff5957,"Electronic Snap Circuits Mini Kits Classpack, ...",NaN,NaN,Toys & Games | Learning & Education | Science ...,NaN,NaN,$99.95,NaN,55324,...,NaN,NaN,NaN,Y,NaN,NaN,99.95,1,Product: Electronic Snap Circuits Mini Kits Cl...,/Users/emandabisrat/Downloads/GRAD SCHOOL/GEN ...
2,2c55cae269aebf53838484b0d7dd931a,3Doodler Create Flexy 3D Printing Filament Ref...,NaN,NaN,Toys & Games | Arts & Crafts | Craft Kits,NaN,NaN,$34.99,NaN,NaN,...,NaN,NaN,NaN,Y,NaN,NaN,34.99,2,Product: 3Doodler Create Flexy 3D Printing Fil...,/Users/emandabisrat/Downloads/GRAD SCHOOL/GEN ...
3,18018b6bc416dab347b1b7db79994afa,Guillow Airplane Design Studio with Travel Cas...,NaN,NaN,Toys & Games | Hobbies | Models & Model Kits |...,NaN,NaN,$28.91,NaN,142,...,NaN,NaN,NaN,Y,NaN,NaN,28.91,3,Product: Guillow Airplane Design Studio with T...,/Users/emandabisrat/Downloads/GRAD SCHOOL/GEN ...
4,e04b990e95bf73bbe6a3fa09785d7cd0,Woodstock- Collage 500 pc Puzzle,NaN,NaN,Toys & Games | Puzzles | Jigsaw Puzzles,NaN,NaN,$17.49,NaN,62151,...,NaN,NaN,NaN,Y,NaN,NaN,17.49,4,Product: Woodstock- Collage 500 pc Puzzle | Ca...,/Users/emandabisrat/Downloads/GRAD SCHOOL/GEN ...


In [19]:
recall_df = evaluate_retrieval(
    store,
    encoder,
    df,
    n_queries=200
)

recall_df

INFO | Evaluating retrieval on 200 synthetic queries …
Eval: 100%|██████████| 200/200 [00:06<00:00, 30.23it/s]
INFO | 
Retrieval Evaluation Results:
INFO |   Recall@ 1 = 0.9950
INFO |   Recall@ 5 = 1.0000
INFO |   Recall@10 = 1.0000


,Recall@1,Recall@5,Recall@10,n_queries
0,0.995,1.0,1.0,200


In [20]:
recall_df.T

,0
Recall@1,0.995
Recall@5,1.000
Recall@10,1.000
n_queries,200.000


In [21]:
def evaluate_mrr(
    store,
    encoder,
    df,
    n_queries=200,
    top_k=10,
):
    sample = df.sample(min(n_queries, len(df)), random_state=42)

    reciprocal_ranks = []

    for _, row in tqdm(sample.iterrows(), total=len(sample)):
        query_emb = encoder.encode_single_text(
            row["combined_description"]
        )

        results = store.query_text(query_emb, top_k=top_k)

        result_ids = [str(r["product_id"]) for r in results]

        try:
            rank = result_ids.index(str(row["product_id"])) + 1
            reciprocal_ranks.append(1.0 / rank)

        except ValueError:
            reciprocal_ranks.append(0.0)

    return float(np.mean(reciprocal_ranks))

In [22]:
from tqdm import tqdm
mrr = evaluate_mrr(store, encoder, df)

print("MRR@10:", round(mrr, 4))

100%|██████████| 200/200 [00:06<00:00, 29.04it/s]

MRR@10: 0.9975


In [23]:
JUDGE_PROMPT = """
You are an evaluator for a product assistant chatbot.

Return ONLY valid JSON:

{{
  "relevance_score": <float>,
  "hallucination_score": <float>,
  "relevance_reason": "<text>",
  "hallucination_reason": "<text>"
}}

User Question: {question}

Retrieved Context:
{context}

Assistant Response:
{response}
"""

In [24]:
TEST_QUERIES = [
    "What are the features of this product?",
    "Is this product waterproof?",
    "What is the battery life?",
    "Can you compare these two products?",
    "What colors does this come in?",
]

In [25]:
def evaluate_response_quality(
    store,
    encoder,
    df,
    n_queries=10,
    top_k=cfg.TOP_K,
):
    sample = df.sample(min(n_queries, len(df)), random_state=99)

    records = []

    for i, (_, row) in enumerate(
        tqdm(sample.iterrows(), total=len(sample))
    ):

        question = TEST_QUERIES[i % len(TEST_QUERIES)]

        product_name = row.get("product_name", "this product")

        full_question = (
            question.replace("this product", product_name)
        )

        query_emb = encoder.encode_single_text(full_question)

        results = store.query_text(query_emb, top_k=top_k)

        context = _format_product_context(results)

        prompt = few_shot_prompt(full_question, context)

        try:
            resp = ollama.chat(
                model=cfg.LLM_MODEL,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
            )

            response_text = resp["message"]["content"]

        except Exception as e:
            print("LLM failed:", e)
            response_text = ""

        judge_input = JUDGE_PROMPT.format(
            question=full_question,
            context=context[:1500],
            response=response_text[:800],
        )

        try:
            judge_resp = ollama.chat(
                model=cfg.LLM_MODEL,
                messages=[
                    {
                        "role": "user",
                        "content": judge_input
                    }
                ],
            )

            raw = judge_resp["message"]["content"]

            raw = re.sub(r"```json|```", "", raw).strip()

            scores = json.loads(raw)

        except Exception as e:
            print("Judge parse failed:", e)

            scores = {
                "relevance_score": None,
                "hallucination_score": None,
                "relevance_reason": "parse error",
                "hallucination_reason": "parse error",
            }

        records.append({
            "product_name": product_name,
            "question": full_question,
            "response": response_text,
            "relevance_score": scores.get("relevance_score"),
            "hallucination_score": scores.get("hallucination_score"),
            "relevance_reason": scores.get("relevance_reason"),
            "hallucination_reason": scores.get("hallucination_reason"),
        })

    return pd.DataFrame(records)

In [26]:
response_df = evaluate_response_quality(
    store,
    encoder,
    df,
    n_queries=10
)

response_df.head()

INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO | HTTP 

,product_name,question,response,relevance_score,hallucination_score,relevance_reason,hallucination_reason
0,Ben 10 Hot Shot Action Figure,What are the features of Ben 10 Hot Shot Actio...,The Ben 10 Hot Shot Action Figure is a highly...,0.8574,0.00,The response is highly relevant to the user's ...,There are no hallucinations in the provided co...
1,Funko Pop! Keychain: Frozen 2 - Elsa,Is Funko Pop! Keychain: Frozen 2 - Elsa waterp...,"No, Funko Pop! Keychain: Frozen 2 - Elsa is n...",0.8582,0.00,The product is a Funko Pop! Keychain and not a...,There is no hallucination in the user question...
2,Great Eastern GE-52716 Dragon Ball Z - Super S...,What is the battery life?,"I'm sorry, but none of the provided products ...",0.0000,0.00,None of the products provided have a specific ...,No relevant information provided.
3,POOF Xtreme Flyerz Vertex 100 Kids Outdoor Plane,Can you compare these two products?,"Sure! The product context provided is for a ""...",0.1200,0.16,The user question is about comparing two produ...,There is no hallucination in this case as the ...
4,3 Wheeled Scooter for Kids - Stand & Cruise Ch...,What colors does this come in?,The Colorations Tempera Paint that is availab...,0.7861,0.00,The product context describes a paint product ...,There is no indication of any other colors bei...


In [27]:
def evaluate_image_retrieval(
    store,
    encoder,
    df,
    n_queries=100,
):

    img_df = df[df["local_image_path"].notna()].copy()

    sample = img_df.sample(
        min(n_queries, len(img_df)),
        random_state=42
    )

    hits_1 = 0
    hits_5 = 0

    for _, row in tqdm(sample.iterrows(), total=len(sample)):

        query = str(row.get("product_name", ""))

        query_emb = encoder.encode_single_text(query)

        results = store.query_image(query_emb, top_k=5)

        result_ids = [r["product_id"] for r in results]

        if str(row["product_id"]) in result_ids[:1]:
            hits_1 += 1

        if str(row["product_id"]) in result_ids[:5]:
            hits_5 += 1

    total = len(sample)

    return {
        "Image Recall@1": hits_1 / total,
        "Image Recall@5": hits_5 / total,
    }

In [28]:
image_metrics = evaluate_image_retrieval(
    store,
    encoder,
    df
)

image_metrics

100%|██████████| 100/100 [00:04<00:00, 21.68it/s]


{'Image Recall@1': 0.62, 'Image Recall@5': 0.85}

In [29]:
valid = response_df.dropna(
    subset=[
        "relevance_score",
        "hallucination_score"
    ]
)

avg_rel = valid["relevance_score"].mean()

avg_hall = valid["hallucination_score"].mean()

print("=" * 50)

print("Recall@K")
print(recall_df)

print("\nMRR@10")
print(round(mrr, 4))

print("\nImage Retrieval")
print(image_metrics)

print("\nResponse Quality")
print("Average Relevance:", round(avg_rel, 4))
print("Average Hallucination:", round(avg_hall, 4))

print("=" * 50)

Recall@K
   Recall@1  Recall@5  Recall@10  n_queries
0     0.995       1.0        1.0        200

MRR@10
0.9975

Image Retrieval
{'Image Recall@1': 0.62, 'Image Recall@5': 0.85}

Response Quality
Average Relevance: 0.6612
Average Hallucination: 0.016
